In [ ]:
#import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#load dataset

df =pd.read_csv('D:\BookAnalysisProject\books_metadata.csv')

In [ ]:
df.shape

In [ ]:
#head()- displays first 5 observarions        head(n)- displays first n observations
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
#check duplicate values
df.nunique()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
#Strip whitespace from text columns

text_cols = ['title', 'subtitle', 'authors', 'publisher', 'description', 'categories', 'language', 'currency', 'buyable', 'search_category']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
#Handling missing data and correct types
df.replace('', pd.NA, inplace=True)

# Convert numeric fields
numeric_cols = ['page_count', 'ratings_count', 'average_rating', 'list_price']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert published_date to datetime and extract year
df['published_date'] = pd.to_datetime(df['published_date'], errors='coerce')
df['year'] = df['published_date'].dt.year

# Drop the original date column
df.drop('published_date', axis=1, inplace=True)


# Handle price outliers
df = df[(df['list_price'] > 0) & (df['list_price'] < df['list_price'].quantile(0.99))]

# Fill numeric NaN values with 0
df[['page_count', 'ratings_count', 'average_rating']] = df[['page_count', 'ratings_count', 'average_rating']].fillna(0)

# Fill missing text columns
df.fillna({'authors': 'Unknown Author', 'publisher': 'Unknown Publisher', 'description': 'No Description', 'categories': 'Uncategorized','language': 'Unknown Language', 'currency': 'Unknown Currency', 'buyable': 'Unknown Buyable Status'}, inplace=True)

In [ ]:
# Drop column, If only 1 values exists
if df['currency'].nunique() == 1: df.drop('currency', axis=1, inplace=True)


In [ ]:
#replace 0s
df['average_rating'] = df['average_rating'].replace(0, df['average_rating'].mean())
df['ratings_count'] = df['ratings_count'].replace(0, df['ratings_count'].median())


In [ ]:
#merging ratings count and average rating into a single popularity score
df['popularity_score'] = df['ratings_count'] * df['average_rating']


In [ ]:
#Drop columns not useful for analysis
df.drop(['preview_link', 'thumbnail', 'info_link', 'search_category'], axis=1, inplace=True)


In [ ]:
# Remove rows where both ISBNs are missing
df = df.dropna(subset=['isbn_10', 'isbn_13'])

#remove dashes, standardize formatting
df['isbn_10'] = df['isbn_10'].astype(str).str.replace('-', '').str.strip()
df['isbn_13'] = df['isbn_13'].astype(str).str.replace('-', '').str.strip()



In [ ]:
# Mapped language codes to full names
language_map = {
    'en': 'English',
    'es': 'Spanish',
    'de': 'German',
    'fr': 'French',
    'pt-BR': 'Portuguese (Brazil)',
    'pt': 'Portuguese',
    'it': 'Italian',
    'zh-CN': 'Chinese',
    'ko': 'Korean',
    'da': 'Danish',
    'fi': 'Finnish',
    'nl': 'Dutch',
    'ro': 'Romanian',
    'ca': 'Catalan',
    'hu': 'Hungarian',
    'te': 'Telugu',
    'cs': 'Czech',
    'tr': 'Turkish',
    'pl': 'Polish'
}
df['language'] = df['language'].map(language_map).fillna('Other')


In [ ]:
# Categorize books based on page_count
def categorize_pages(pages):
    if pages < 100: return 'Short ( <100 pages )'
    elif pages <= 300: return 'Medium (100-300 pages)'
    else: return 'Long ( >300 pages )'

df['pages_group'] = df['page_count'].apply(categorize_pages)

In [ ]:
# Count number of authors
df['num_authors'] = df['authors'].apply(lambda x: len(str(x).split(',')))
display(df[['title', 'page_count', 'pages_group', 'authors', 'num_authors']].head(10))


In [ ]:
# ==========================================
# DESCRIPTIVE STATISTICS
# ==========================================

numeric_cols = [
    'page_count',
    'list_price',
    'popularity_score',
    'num_authors',
    'year'
]

df[numeric_cols].describe()

In [ ]:
for col in numeric_cols:
    print(f"\n{col}")
    print("Mean :", df[col].mean())
    print("Median :", df[col].median())
    print("Std :", df[col].std())
    print("Variance :", df[col].var())

In [ ]:
# ==========================================
# HYPOTHESIS TESTING
# ==========================================

from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency

In [ ]:
#Test 1: Does page count affect popularity?         #T-Test

median_popularity = df['popularity_score'].median()

popular = df[df['popularity_score'] >= median_popularity]['page_count']
less_popular = df[df['popularity_score'] < median_popularity]['page_count']

t_stat,p_value = ttest_ind(popular,less_popular,nan_policy='omit')

print("T Statistic:",t_stat)
print("P Value:",p_value)

In [ ]:
if p_value < 0.05:
    print("Significant difference exists")
else:
    print("No significant difference")

In [ ]:
#Popularity across categories                 #ANOVA test

top_categories = (
    df['categories']
    .value_counts()
    .head(5)
    .index
)

groups = []

for category in top_categories:
    groups.append(
        df[df['categories']==category]['popularity_score']
    )

f_stat,p_value = f_oneway(*groups)

print("F Statistic:",f_stat)
print("P Value:",p_value)



In [ ]:
if p_value < 0.05:
    print("Reject H0")
    print("Popularity differs significantly across categories")
else:
    print("Fail to reject H0")

In [ ]:
#Chi-Square 

top_df = df[
    df['categories'].isin(top_categories)
]

contingency_table = pd.crosstab(
    top_df['language'],
    top_df['categories']
)

chi2,p,dof,expected = chi2_contingency(
    contingency_table
)

print("Chi Square:",chi2)
print("P Value:",p)

In [ ]:
if p_value < 0.05:
    print("Reject H0")
    print("Language and category are associated")
else:
    print("Fail to reject H0")

In [ ]:
# Exploratory Data Analysis (EDA)

In [ ]:
# ==========================================
# UNIVARIATE ANALYSIS
# ==========================================

#Popularity Distribution

sns.histplot(df['popularity_score'],kde=True)
plt.title('Popularity Score Distribution')
plt.show()

In [ ]:
#count of books by language 
language_counts = df['language'].value_counts()
print(language_counts)

In [ ]:
#Page Count Distribution

sns.histplot(df['page_count'],kde=True)
plt.show()

In [ ]:
#Book Length Categories

sns.countplot(
    y=df['pages_group'],
    order=df['pages_group'].value_counts().index
)
plt.show()

In [ ]:
#Most popular book categories 
top_categories = df['categories'].value_counts().head(10)
top_categories.plot(kind='bar', figsize=(10,5))
plt.title("Top 10 Book Categories")
plt.ylabel("Number of Books")
plt.show()


In [ ]:
# ==========================================
# BIVARIATE ANALYSIS
# ==========================================

#Top Categories by Popularity Score 

top_categories = df.groupby('categories')['popularity_score'].sum().sort_values(ascending=False).head(10)
print(top_categories)

In [ ]:
#Year vs Popularity

sns.scatterplot(
    data=df,
    x='year',
    y='popularity_score'
)
plt.show()

In [ ]:
#Category vs Popularity

top10 = (
    df.groupby('categories')['popularity_score']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

top10.plot(kind='bar')
plt.show()

In [ ]:
#year wise publication trend
publication_trend = df['year'].value_counts().sort_index()
plt.figure(figsize=(12,6))
plt.plot(publication_trend.index, publication_trend.values)
plt.title("Books publications Over the Years")    
plt.xlabel("Year")
plt.ylabel("Number of Publications")    
plt.show()

In [ ]:
#author popularity
top_authors = df.groupby('authors')['popularity_score'].sum().sort_values(ascending=False).head(10)
print(top_authors)
top_authors.plot(kind='bar', figsize=(10,5))

In [ ]:
#Multivariate: Popularity + Page Count + Number of Authors

plt.figure(figsize=(9,5))
sns.scatterplot(x=df['page_count'], y=df['popularity_score'], hue=df['num_authors'], palette='viridis')
plt.title("Popularity Based on Page Count & Number of Authors")
plt.xlabel("Page Count")
plt.ylabel("Popularity Score")
plt.legend(title="Number of Authors")
plt.show()


In [ ]:
#correlation analysis
corr_cols = [
    'page_count',
    'list_price',
    'popularity_score',
    'num_authors',
    'year'
]

sns.heatmap(
    df[corr_cols].corr(),
    annot=True
)
plt.show()